# Random Forest Regression

In the previous notebook, we built and tuned a Decision Tree Regressor.

Our tuned decision tree achieved a test R² of approximately 0.728.

However, a single decision tree can be sensitive to the particular training data. Small changes in the training data can sometimes produce a different tree structure and therefore different predictions.

Random Forest is an ensemble learning method that addresses this limitation by combining predictions from many decision trees.

In this notebook, we will:

1. Understand why a single decision tree can be unstable.
2. Understand the idea of ensemble learning.
3. Learn how Random Forest combines multiple decision trees.
4. Understand bootstrap sampling.
5. Understand random feature selection.
6. Build a small Random Forest conceptually.
7. Train a `RandomForestRegressor` using Scikit-learn.
8. Compare it with our tuned Decision Tree.
9. Tune the Random Forest using cross-validation.

## Why Use More Than One Decision Tree?

A decision tree learns its structure from the training data.

The training process determines:

- which feature is selected for a split,
- which threshold is selected,
- which samples reach each leaf,
- and therefore the prediction made by each leaf.

Because the tree depends on the training data, changing the data can change the resulting tree.

This means that a single decision tree can have relatively high variance.

Random Forest reduces this problem by training many different decision trees and combining their predictions.

## Ensemble Learning

An **ensemble** is a machine learning model that combines multiple individual models.

Instead of relying on one model:

$$
\text{Prediction} = \text{Model}_1(X)
$$

we can combine several models:

$$
\text{Prediction}
=
\text{combine}(
\text{Model}_1(X),
\text{Model}_2(X),
\dots,
\text{Model}_T(X)
)
$$

Random Forest is an ensemble of decision trees.

For regression, the predictions of the individual trees are averaged:

$$
\hat{y}
=
\frac{1}{T}
\sum_{t=1}^{T}
\hat{y}_t
$$

where:

- $T$ is the number of trees.
- $\hat{y}_t$ is the prediction made by tree $t$.
- $\hat{y}$ is the final Random Forest prediction.

The central idea is therefore:

> Instead of trusting one decision tree, combine the predictions of many trees.

## A Simple Random Forest Example

Suppose five decision trees make the following predictions for one house:

$$
2.0,\ 2.4,\ 2.2,\ 2.6,\ 2.3
$$

The Random Forest averages these predictions:

$$
\hat y =
\frac{2.0+2.4+2.2+2.6+2.3}{5}
=2.3
$$

Therefore, the final prediction of the forest is 2.3.

The individual trees may disagree, but averaging their predictions can produce a more stable prediction.

## Bootstrap Sampling

If we trained every decision tree on exactly the same training data, the trees would tend to learn very similar structures.

Random Forest avoids this by giving each tree a different training dataset.

It creates these datasets using **bootstrap sampling**.

Bootstrap sampling means repeatedly selecting samples from the original training data **with replacement**.

For example, suppose our training dataset contains only:

$$
[A,\ B,\ C,\ D,\ E]
$$

A bootstrap sample could be:

$$
[A,\ C,\ C,\ E,\ A]
$$

Notice that:

- `A` appears twice.
- `C` appears twice.
- `B` and `D` were not selected.
- The bootstrap sample still contains 5 observations.

Another tree might receive:

$$
[B,\ D,\ A,\ D,\ E]
$$

This gives the trees different training data, causing them to learn different structures.

In [3]:
import numpy as np

data = np.array(["A", "B", "C", "D", "E"])

bootstrap_sample = np.random.choice(
    data,
    size = len(data),
    replace=True
)

print("Original data:", data)
print("Bootstrap sample:", bootstrap_sample)

Original data: ['A' 'B' 'C' 'D' 'E']
Bootstrap sample: ['C' 'E' 'B' 'A' 'E']


## Bootstrap Samples in Random Forest

For our California Housing training data, we have 16,512 training observations.

Random Forest can create a bootstrap sample of approximately the same size for each tree by sampling the training observations with replacement.

For example:

$$
\text{Training data}
\rightarrow
\begin{cases}
\text{Bootstrap sample 1} \rightarrow \text{Tree 1}\\
\text{Bootstrap sample 2} \rightarrow \text{Tree 2}\\
\text{Bootstrap sample 3} \rightarrow \text{Tree 3}\\
\vdots\\
\text{Bootstrap sample T} \rightarrow \text{Tree T}
\end{cases}
$$

Because the samples are randomly generated, the trees receive different training data and can therefore learn different structures.

This is one of the sources of randomness in a Random Forest.

## Random Feature Selection

Our California Housing dataset has 8 features:

- `MedInc`
- `HouseAge`
- `AveRooms`
- `AveBedrms`
- `Population`
- `AveOccup`
- `Latitude`
- `Longitude`

If every tree could always consider all 8 features when making every split, the trees could still become quite similar, especially when some features are much more useful than others.

Random Forest introduces another source of randomness.

When a tree is deciding how to split a node, it considers only a randomly selected subset of the available features.

For example, instead of considering all 8 features, a particular split might consider:

$$
\{\text{MedInc},\text{AveRooms},\text{Latitude}\}
$$

Another split might consider:

$$
\{\text{HouseAge},\text{Population},\text{Longitude}\}
$$

The subset is selected randomly.

Therefore, different trees can be encouraged to learn different patterns from the data.

### A Small Example

Suppose a dataset contains four features:

$$
A,\ B,\ C,\ D
$$

At a particular split, different trees might receive different random subsets:

- Tree 1 → `A, C`
- Tree 2 → `B, D`
- Tree 3 → `A, B`

Each tree must choose its split from the features available to it.

This means that even if one feature is very strong, it may not always be available at every split.

This encourages the trees to become less correlated with one another.

## How Random Forest Works

A Random Forest combines two sources of randomness.

### 1. Bootstrap Sampling

Each tree receives a different bootstrap sample of the training data.

The samples are selected with replacement.

### 2. Random Feature Selection

When splitting a node, the tree considers only a randomly selected subset of the available features.

These two mechanisms create a collection of different decision trees.

Each tree makes its own prediction.

For regression, the Random Forest averages these predictions:

$$
\hat{y}
=
\frac{1}{T}
\sum_{t=1}^{T}\hat{y}_t
$$

Therefore, the overall process is:

$$
\text{Training Data}
\rightarrow
\text{Random Samples}
\rightarrow
\text{Different Trees}
\rightarrow
\text{Individual Predictions}
\rightarrow
\text{Average}
\rightarrow
\text{Final Prediction}
$$

## A Small Forest

Before using Scikit-learn's `RandomForestRegressor`, we will simulate the basic idea using a small dataset.

The purpose is not to reproduce the complete Random Forest algorithm ourselves.

Instead, we want to see the central mechanism:

$$
\text{Multiple Trees}
\rightarrow
\text{Multiple Predictions}
\rightarrow
\text{Average}
$$

In [1]:
import numpy as np

X_small = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [6],
    [7],
    [8]
])

y_small = np.array([
    2,
    4,
    5,
    8,
    9,
    12,
    13,
    16,
])

In [4]:
rng = np.random.default_rng(42)

indices_1 = rng.choice(len(X_small), size=len(X_small), replace=True)
indices_2 = rng.choice(len(X_small), size=len(X_small), replace=True)
indices_3 = rng.choice(len(X_small), size=len(X_small), replace=True)

X1 = X_small[indices_1]
y1 = y_small[indices_1]

X2 = X_small[indices_2]
y2 = y_small[indices_2]

X3 = X_small[indices_3]
y3 = y_small[indices_3]

print("Tree 1 indices:", indices_1)
print("Tree 2 indices:", indices_2)
print("Tree 3 indices:", indices_3)

Tree 1 indices: [0 6 5 3 3 6 0 5]
Tree 2 indices: [1 0 4 7 5 6 5 6]
Tree 3 indices: [4 1 6 3 4 2 1 7]


In [5]:
from sklearn.tree import DecisionTreeRegressor

tree_1 = DecisionTreeRegressor(
    max_depth=3,
    random_state=1
)

tree_2 = DecisionTreeRegressor(
    max_depth=3,
    random_state=2
)

tree_3 = DecisionTreeRegressor(
    max_depth=3,
    random_state=3
)

tree_1.fit(X1, y1)
tree_2.fit(X2, y2)
tree_3.fit(X3, y3)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",3
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",3
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_

In [6]:
sample = np.array([[5.5]])

pred1 = tree_1.predict(sample)[0]
pred2 = tree_2.predict(sample)[0]
pred3 = tree_3.predict(sample)[0]

print("Tree 1 prediction:", pred1)
print("Tree 2 prediction:", pred2)
print("Tree 3 prediction:", pred3)

Tree 1 prediction: 12.0
Tree 2 prediction: 9.0
Tree 3 prediction: 9.0


In [7]:
forest_prediction = np.mean([
    pred1,
    pred2,
    pred3
])

print("Random Forest prediction:", forest_prediction)

Random Forest prediction: 10.0


## What We Just Demonstrated

We created three decision trees using different bootstrap samples.

Because the trees received different training observations, they could learn different structures and produce different predictions.

We then averaged their predictions.

This is the central idea behind Random Forest regression.

A Random Forest does not necessarily make every individual tree better.

Instead, it combines many different trees so that the final prediction can be more stable than the prediction of a single tree.

The next step is to replace our manually constructed collection of trees with Scikit-learn's `RandomForestRegressor` and see whether this ensemble improves upon our tuned decision tree.